# NICE-RAG — Private Synthetic Validation

This notebook validates only the provider-free synthetic NICE-RAG release. It clones the reviewed public source, installs declared Python packages inside Kaggle, runs compile/tests and bounded synthetic CLI checks, writes one sanitized evidence file, and stops.

**Closed gates:** NICE PDFs, embedding-model downloads, Chroma construction, Groq/API execution, patient data, live clinical traces, Hugging Face, deployment, and publication. This is research information only, not clinical decision support.

In [ ]:
import json
import os
import platform
import re
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path

REPOSITORY = 'https://github.com/ajinkya-awari/-nice-rag.git'
ROOT = Path('/kaggle/working/nice_rag')
EVIDENCE_PATH = Path('/kaggle/working/nice_rag_synthetic_evidence.json')

def run_checked(command, label, cwd=None):
    result = subprocess.run(command, cwd=cwd, capture_output=True, text=True)
    print(f'--- {label} (exit={result.returncode}) ---')
    if result.stdout:
        print(result.stdout.rstrip())
    if result.stderr:
        print(result.stderr.rstrip())
    if result.returncode != 0:
        raise RuntimeError(f'{label} failed with exit code {result.returncode}')
    return result

print('Python:', sys.version)
print('Platform:', platform.platform())
print('CWD:', os.getcwd())
input_root = Path('/kaggle/input')
inputs = sorted(path.name for path in input_root.iterdir()) if input_root.is_dir() else []
print('Attached inputs:', inputs or '(none)')
if inputs:
    raise RuntimeError('Synthetic validation requires no attached datasets or models')

nvidia_smi = shutil.which('nvidia-smi')
if nvidia_smi:
    gpu_probe = subprocess.run(
        [nvidia_smi, '--query-gpu=name', '--format=csv,noheader'],
        capture_output=True, text=True
    )
    gpu_name = gpu_probe.stdout.strip() or 'none'
else:
    gpu_name = 'none (nvidia-smi unavailable)'
print('GPU visible:', gpu_name)

if ROOT.exists():
    shutil.rmtree(ROOT)
run_checked(['git', 'clone', '--depth', '1', REPOSITORY, str(ROOT)], 'git clone')
revision = run_checked(['git', 'rev-parse', 'HEAD'], 'source revision', ROOT).stdout.strip()
print('Source revision:', revision)

In [ ]:
install_result = run_checked(
    [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-q', '-r', 'requirements.txt'],
    'dependency installation', ROOT
)
package_names = [
    'langchain', 'langchain-community', 'langchain-core',
    'langchain-text-splitters', 'langchain-groq', 'langchain-huggingface',
    'chromadb', 'sentence-transformers', 'pypdf', 'gradio', 'pyyaml', 'pytest'
]
dependency_versions = {name: metadata.version(name) for name in package_names}
print(json.dumps(dependency_versions, indent=2, sort_keys=True))

In [ ]:
compile_result = run_checked(
    [sys.executable, '-m', 'compileall', 'src', 'tests'],
    'compileall', ROOT
)
pytest_result = run_checked(
    [sys.executable, '-m', 'pytest', '-q'],
    'pytest', ROOT
)
pass_match = re.search(r'(\d+) passed', pytest_result.stdout)
if not pass_match:
    raise RuntimeError('pytest output did not contain a passed-test count')
pytest_passed = int(pass_match.group(1))

In [ ]:
cpu_result = run_checked(
    [sys.executable, 'run.py', '--cpu-smoke', '--documents', '1000', '--repeats', '1'],
    'bounded CPU smoke', ROOT
)
cpu_smoke = {}
for line in cpu_result.stdout.splitlines():
    if '=' in line:
        key, value = line.split('=', 1)
        cpu_smoke[key.strip()] = value.strip()
citation_validity = cpu_smoke.get('all_citations_valid') == 'True'
if not citation_validity:
    raise RuntimeError('CPU smoke citation-validity gate failed')
if int(cpu_smoke.get('max_passages', '999')) > 3:
    raise RuntimeError('CPU smoke exceeded the three-passage cap')

scenario_result = run_checked(
    [sys.executable, 'run.py', '--list-scenarios'],
    'scenario listing', ROOT
)
scenario_count = scenario_result.stdout.count('gated_no_live_trace')
if scenario_count != 5:
    raise RuntimeError(f'Expected five gated scenarios, found {scenario_count}')

In [ ]:
restricted_suffixes = {'.pdf', '.bin', '.db', '.jsonl', '.parquet', '.safetensors', '.sqlite', '.onnx', '.pt', '.pth', '.pkl'}
restricted_names = {'.env', 'credentials.json', 'credentials.yaml', 'secrets.json'}
restricted_artifacts = []
for path in ROOT.rglob('*'):
    if not path.is_file() or any(part in {'.git', '__pycache__', '.pytest_cache'} for part in path.parts):
        continue
    if path.name.casefold() in restricted_names or path.suffix.casefold() in restricted_suffixes:
        restricted_artifacts.append(path.relative_to(ROOT).as_posix())
if restricted_artifacts:
    raise RuntimeError(f'Restricted artifacts found: {restricted_artifacts}')

evidence = {
    'project': 'NICE-RAG',
    'kernel': 'ajinkya1225/19-nice-rag-validation',
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'python_version': sys.version,
    'platform': platform.platform(),
    'dependency_versions': dependency_versions,
    'gpu_visible': gpu_name,
    'source_revision': revision,
    'compileall_exit_code': compile_result.returncode,
    'pytest_exit_code': pytest_result.returncode,
    'pytest_passed': pytest_passed,
    'cpu_smoke': cpu_smoke,
    'citation_validity': citation_validity,
    'scenario_count': scenario_count,
    'restricted_artifacts': restricted_artifacts,
    'skipped_gates': [
        'NICE PDFs', 'embedding model', 'Chroma', 'Groq/API',
        'patient data', 'live traces', 'Hugging Face', 'deployment', 'publication'
    ]
}
with EVIDENCE_PATH.open('w', encoding='utf-8') as handle:
    json.dump(evidence, handle, indent=2, sort_keys=True)
print(json.dumps(evidence, indent=2, sort_keys=True))
print('Evidence written to:', EVIDENCE_PATH)
print('STOP: all external data, model, provider, clinical, and deployment gates remain closed.')